# Mobile Money Agent Network Data Clean-Up

## Power Learn Project — Data Analytics Assignment 4

This project cleans and validates a synthetic mobile money agent transaction dataset.

### Objectives

- Identify data-quality problems.
- Clean transaction data using a reusable Python function.
- Merge transactions with agent reference data.
- Track cleaning actions using a data-quality audit log.
- Validate the cleaned dataset.
- Summarize transactions by county.
- Analyze transactions using a weekly pivot table.
- Produce a Data Quality Report covering:
  - Completeness
  - Validity
  - Accuracy
  - Consistency
  - Uniqueness
  - Timeliness

In [1]:
import pandas as pd
import numpy as np

from cleaning_pipeline import clean_agent_data

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
txns = pd.read_csv("outputs/raw_transactions.csv")
agents = pd.read_csv("outputs/agents.csv")

print("Transaction rows:", len(txns))
print("Agent rows:", len(agents))

print("\nTransaction data:")
display(txns.head())

print("\nAgent data:")
display(agents.head())

Transaction rows: 101
Agent rows: 21

Transaction data:


,txn_id,agent_id,txn_type,amount,txn_date
0,TXN0001,AG004,Withdrawal,7660,2026-01-01
1,TXN0002,AG018,Deposit,5378,2026-01-02
2,TXN0003,AG014,Deposit,4959,2026-01-03
3,TXN0004,AG004,Withdrawal,8570,2026-01-04
4,TXN0005,AG001,Deposit,12709,2026-01-05



Agent data:


,agent_id,county
0,AG001,Nairobi
1,AG002,Mombasa
2,AG003,Kisumu
3,AG004,Nakuru
4,AG005,Laikipia


In [3]:
print("TRANSACTION DATA TYPES")
print("=" * 50)
print(txns.dtypes)

print("\nMISSING VALUES")
print("=" * 50)
display(txns.isna().sum().to_frame("missing_rows"))

print("\nDUPLICATE TRANSACTION IDs")
print("=" * 50)
print(txns["txn_id"].duplicated().sum())

print("\nTRANSACTION TYPES")
print("=" * 50)
print(txns["txn_type"].value_counts(dropna=False))

print("\nDUPLICATE AGENT IDs")
print("=" * 50)
print(agents["agent_id"].duplicated().sum())

TRANSACTION DATA TYPES
txn_id      str
agent_id    str
txn_type    str
amount      str
txn_date    str
dtype: object

MISSING VALUES


,missing_rows
txn_id,0
agent_id,0
txn_type,0
amount,2
txn_date,1



DUPLICATE TRANSACTION IDs
1

TRANSACTION TYPES
txn_type
Deposit        58
Withdrawal     40
Transfer        1
deposit         1
 WITHDRAWAL     1
Name: count, dtype: int64

DUPLICATE AGENT IDs
1


In [4]:
clean_df, log = clean_agent_data(txns, agents)

print("CLEANING COMPLETED")
print("=" * 50)
print("Rows before cleaning:", len(txns))
print("Rows after cleaning:", len(clean_df))
print("Rows removed:", len(txns) - len(clean_df))

CLEANING COMPLETED
Rows before cleaning: 101
Rows after cleaning: 95
Rows removed: 6


In [5]:
display(log)

,issue,dimension,column,rows affected
0,Duplicate transaction IDs removed,Uniqueness,txn_id,1
1,Invalid transaction types removed,Validity,txn_type,1
2,Non-numeric amounts converted to missing,Validity,amount,1
3,Missing amounts imputed using the median,Completeness,amount,3
4,Invalid dates converted to missing,Validity,txn_date,3
5,Transactions with missing dates removed,Completeness,txn_date,4
6,Amounts outside the IQR fences flagged as outl...,Accuracy,amount,1
7,Duplicate agent reference IDs removed,Uniqueness,agent_id,1
8,Transactions without a matching agent,Consistency,agent_id,1


In [6]:
audit_table = log[
    ["issue", "dimension", "column", "rows affected"]
].copy()

display(audit_table)

,issue,dimension,column,rows affected
0,Duplicate transaction IDs removed,Uniqueness,txn_id,1
1,Invalid transaction types removed,Validity,txn_type,1
2,Non-numeric amounts converted to missing,Validity,amount,1
3,Missing amounts imputed using the median,Completeness,amount,3
4,Invalid dates converted to missing,Validity,txn_date,3
5,Transactions with missing dates removed,Completeness,txn_date,4
6,Amounts outside the IQR fences flagged as outl...,Accuracy,amount,1
7,Duplicate agent reference IDs removed,Uniqueness,agent_id,1
8,Transactions without a matching agent,Consistency,agent_id,1


In [7]:
validation = pd.DataFrame({
    "metric": [
        "Rows before cleaning",
        "Rows after cleaning",
        "Rows removed",
        "Duplicate transaction IDs",
        "Transactions with unknown agents",
        "Missing counties"
    ],
    "value": [
        len(txns),
        len(clean_df),
        len(txns) - len(clean_df),
        clean_df["txn_id"].duplicated().sum(),
        clean_df["county"].isna().sum(),
        clean_df["county"].isna().sum()
    ]
})

display(validation)

,metric,value
0,Rows before cleaning,101
1,Rows after cleaning,95
2,Rows removed,6
3,Duplicate transaction IDs,0
4,Transactions with unknown agents,1
5,Missing counties,1


In [8]:
print("Transaction types after cleaning:")
print(clean_df["txn_type"].value_counts())

assert clean_df["txn_type"].isin(
    ["Deposit", "Withdrawal"]
).all()

print("\nTransaction type validation PASSED.")

Transaction types after cleaning:
txn_type
Deposit       56
Withdrawal    39
Name: count, dtype: Int64

Transaction type validation PASSED.


In [9]:
print("Amount data type:")
print(clean_df["amount"].dtype)

print("\nNumber of imputed amounts:")
print(clean_df["amount_imputed"].sum())

assert pd.api.types.is_numeric_dtype(clean_df["amount"])

print("\nAmount validation PASSED.")

Amount data type:
Int64

Number of imputed amounts:
3

Amount validation PASSED.


In [10]:
print("Transaction date data type:")
print(clean_df["txn_date"].dtype)

print("\nDate range:")
print("Minimum:", clean_df["txn_date"].min())
print("Maximum:", clean_df["txn_date"].max())

assert pd.api.types.is_datetime64_any_dtype(clean_df["txn_date"])

print("\nDate validation PASSED.")

Transaction date data type:
datetime64[us]

Date range:
Minimum: 2026-01-01 00:00:00
Maximum: 2026-04-10 00:00:00

Date validation PASSED.


In [11]:
duplicate_count = clean_df["txn_id"].duplicated().sum()

print("Duplicate transaction IDs:", duplicate_count)

assert duplicate_count == 0

print("Uniqueness validation PASSED.")

Duplicate transaction IDs: 0
Uniqueness validation PASSED.


In [12]:
required_flags = [
    "amount_imputed",
    "is_outlier"
]

for column in required_flags:
    assert column in clean_df.columns

print("Required flags are present:")
print(required_flags)

print("\nFlag validation PASSED.")

Required flags are present:
['amount_imputed', 'is_outlier']

Flag validation PASSED.


In [13]:
county_summary = (
    clean_df.groupby("county", dropna=False)
    .agg(
        transactions=("txn_id", "count"),
        total_amount=("amount", "sum"),
        average_amount=("amount", "mean"),
        outliers=("is_outlier", "sum")
    )
    .sort_values("total_amount", ascending=False)
)

display(county_summary)

,transactions,total_amount,average_amount,outliers
county,,,,
Kisumu,5,84109,16821.8,1
Machakos,8,63123,7890.375,0
Nakuru,8,61055,7631.875,0
Laikipia,9,56661,6295.666667,0
Garissa,6,51119,8519.833333,0
Kilifi,5,46691,9338.2,0
Narok,8,46569,5821.125,0
Kajiado,4,37998,9499.5,0
Nyeri,7,37866,5409.428571,0


In [17]:
clean_df["week"] = (
    clean_df["txn_date"]
    .dt.to_period("W")
    .astype(str)
)

weekly_pivot = pd.pivot_table(
    clean_df,
    index="week",
    columns="txn_type",
    values="amount",
    aggfunc="sum",
    fill_value=0
)

display(weekly_pivot)

txn_type,Deposit,Withdrawal
week,,
2025-12-29/2026-01-04,10337,16230
2026-01-05/2026-01-11,41985,1561
2026-01-12/2026-01-18,26295,40191
2026-01-19/2026-01-25,34034,9039
2026-01-26/2026-02-01,42140,7572
2026-02-02/2026-02-08,14570,25663
2026-02-09/2026-02-15,77068,6581
2026-02-16/2026-02-22,30660,19751
2026-02-23/2026-03-01,42394,33337


In [20]:
final_checks = pd.DataFrame({
    "check": [
        "No duplicate txn_id",
        "Only Deposit and Withdrawal",
        "Amount is numeric",
        "Transaction date is valid",
        "amount_imputed flag exists",
        "is_outlier flag exists",
        "County column exists",
        "Unknown agents identified and documented"
    ],
    "passed": [
        not clean_df["txn_id"].duplicated().any(),
        clean_df["txn_type"].isin(
            ["Deposit", "Withdrawal"]
        ).all(),
        pd.api.types.is_numeric_dtype(
            clean_df["amount"]
        ),
        pd.api.types.is_datetime64_any_dtype(
            clean_df["txn_date"]
        ),
        "amount_imputed" in clean_df.columns,
        "is_outlier" in clean_df.columns,
        "county" in clean_df.columns,
        "county" in clean_df.columns
    ]
})

display(final_checks)

,check,passed
0,No duplicate txn_id,True
1,Only Deposit and Withdrawal,True
2,Amount is numeric,True
3,Transaction date is valid,True
4,amount_imputed flag exists,True
5,is_outlier flag exists,True
6,County column exists,True
7,Unknown agents identified and documented,True


In [21]:
if final_checks["passed"].all():
    print("ALL FINAL DATA-QUALITY CHECKS PASSED.")
else:
    print("Some validation checks need attention.")
    display(final_checks[final_checks["passed"] == False])

ALL FINAL DATA-QUALITY CHECKS PASSED.


In [22]:
unknown_agents = clean_df[clean_df["county"].isna()].copy()

print("Number of transactions with unknown agents:", len(unknown_agents))

if len(unknown_agents) > 0:
    print("\nUnknown agent transactions:")
    display(
        unknown_agents[
            ["txn_id", "agent_id", "txn_type", "amount", "txn_date"]
        ]
    )
else:
    print("No unknown agents found.")

Number of transactions with unknown agents: 1

Unknown agent transactions:


,txn_id,agent_id,txn_type,amount,txn_date
66,TXN0071,AG999,Deposit,7803,2026-03-12


In [23]:
county_summary = (
    clean_df.groupby("county", dropna=False)
    .agg(
        transactions=("txn_id", "count"),
        total_amount=("amount", "sum"),
        average_amount=("amount", "mean"),
        outliers=("is_outlier", "sum")
    )
    .sort_values("total_amount", ascending=False)
)

display(county_summary)

,transactions,total_amount,average_amount,outliers
county,,,,
Kisumu,5,84109,16821.8,1
Machakos,8,63123,7890.375,0
Nakuru,8,61055,7631.875,0
Laikipia,9,56661,6295.666667,0
Garissa,6,51119,8519.833333,0
Kilifi,5,46691,9338.2,0
Narok,8,46569,5821.125,0
Kajiado,4,37998,9499.5,0
Nyeri,7,37866,5409.428571,0


In [24]:
clean_df["week"] = (
    clean_df["txn_date"]
    .dt.to_period("W")
    .astype(str)
)

weekly_pivot = pd.pivot_table(
    clean_df,
    index="week",
    columns="txn_type",
    values="amount",
    aggfunc="sum",
    fill_value=0
)

display(weekly_pivot)

txn_type,Deposit,Withdrawal
week,,
2025-12-29/2026-01-04,10337,16230
2026-01-05/2026-01-11,41985,1561
2026-01-12/2026-01-18,26295,40191
2026-01-19/2026-01-25,34034,9039
2026-01-26/2026-02-01,42140,7572
2026-02-02/2026-02-08,14570,25663
2026-02-09/2026-02-15,77068,6581
2026-02-16/2026-02-22,30660,19751
2026-02-23/2026-03-01,42394,33337


In [25]:
export_df = clean_df.copy()

export_df["txn_date"] = export_df["txn_date"].dt.strftime(
    "%Y-%m-%d"
)

export_df.to_csv(
    "outputs/agent_transactions_clean.csv",
    index=False
)

print("Clean dataset exported successfully.")
print("Rows exported:", len(export_df))
print("File: outputs/agent_transactions_clean.csv")

Clean dataset exported successfully.
Rows exported: 95
File: outputs/agent_transactions_clean.csv


In [26]:
final_csv = pd.read_csv(
    "outputs/agent_transactions_clean.csv"
)

print("Exported rows:", len(final_csv))
print("Exported columns:")
print(final_csv.columns.tolist())

display(final_csv.head(10))

Exported rows: 95
Exported columns:
['txn_id', 'agent_id', 'txn_type', 'amount', 'txn_date', 'amount_imputed', 'is_outlier', 'county', 'week']


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county,week
0,TXN0001,AG004,Withdrawal,7660,2026-01-01,False,False,Nakuru,2025-12-29/2026-01-04
1,TXN0002,AG018,Deposit,5378,2026-01-02,False,False,Kilifi,2025-12-29/2026-01-04
2,TXN0003,AG014,Deposit,4959,2026-01-03,False,False,Garissa,2025-12-29/2026-01-04
3,TXN0004,AG004,Withdrawal,8570,2026-01-04,False,False,Nakuru,2025-12-29/2026-01-04
4,TXN0005,AG001,Deposit,12709,2026-01-05,False,False,Nairobi,2026-01-05/2026-01-11
5,TXN0006,AG004,Deposit,5000,2026-01-06,False,False,Nakuru,2026-01-05/2026-01-11
6,TXN0007,AG016,Deposit,13414,2026-01-07,False,False,Uasin Gishu,2026-01-05/2026-01-11
7,TXN0008,AG011,Withdrawal,1561,2026-01-08,False,False,Nyeri,2026-01-05/2026-01-11
8,TXN0009,AG004,Deposit,5367,2026-01-09,False,False,Nakuru,2026-01-05/2026-01-11
9,TXN0010,AG008,Deposit,469,2026-01-10,False,False,Kiambu,2026-01-05/2026-01-11


In [27]:
print("=" * 60)
print("FINAL OUTPUT QUALITY CHECK")
print("=" * 60)

checks = {
    "No duplicate txn_id": not final_csv["txn_id"].duplicated().any(),
    "Valid transaction types": final_csv["txn_type"].isin(
        ["Deposit", "Withdrawal"]
    ).all(),
    "Amount is numeric": pd.api.types.is_numeric_dtype(
        final_csv["amount"]
    ),
    "Dates are ISO formatted": final_csv["txn_date"].str.match(
        r"^\d{4}-\d{2}-\d{2}$"
    ).all(),
    "amount_imputed exists": "amount_imputed" in final_csv.columns,
    "is_outlier exists": "is_outlier" in final_csv.columns,
    "county exists": "county" in final_csv.columns
}

for check, result in checks.items():
    print(f"{check}: {'PASS' if result else 'FAIL'}")

print("=" * 60)

assert all(checks.values())

print("FINAL OUTPUT VALIDATION PASSED.")

FINAL OUTPUT QUALITY CHECK
No duplicate txn_id: PASS
Valid transaction types: PASS
Amount is numeric: PASS
Dates are ISO formatted: PASS
amount_imputed exists: PASS
is_outlier exists: PASS
county exists: PASS
FINAL OUTPUT VALIDATION PASSED.


# Data Quality Report

## Completeness

The dataset was checked for missing transaction amounts and dates. Missing amounts were identified using the `amount_imputed` flag and replaced using the median valid amount. Transactions with unusable dates were removed because dates are required for reliable time-based analysis.

## Validity

Transaction types were standardized and restricted to `Deposit` and `Withdrawal`. Transaction amounts were converted to numeric values, while invalid numeric values were converted to missing values and handled through the imputation process. Transaction dates were converted to datetime values.

## Accuracy

Transaction amounts were assessed using the Interquartile Range (IQR) method. Values outside the calculated IQR fences were flagged using `is_outlier`. Outliers were retained rather than automatically deleted so that they remain available for investigation.

## Consistency

Agent IDs were standardized by removing unnecessary whitespace. Transactions were then merged with the agent reference data using a left join. Transactions that did not have a matching agent were identified and reported as unknown agents.

## Uniqueness

Transaction IDs were checked for duplicates and duplicate transaction records were removed. Agent reference IDs were also checked for duplicates before the merge to prevent duplicate reference records from creating multiple transaction rows.

## Timeliness

Transaction dates were converted to a consistent datetime representation and exported using the ISO `YYYY-MM-DD` format. Standardized dates allow reliable chronological and weekly analysis.

## Overall Assessment

The cleaning pipeline successfully transforms the raw mobile money transaction data into an analytical dataset while maintaining an audit trail. Data-quality problems are documented rather than hidden, and important transformations are represented through explicit quality flags such as `amount_imputed` and `is_outlier`.